# 14 · Frozen classifier and prospective prediction ledger

**Purpose:** Refit the preselected Logistic L2 (`C=0.10`) on all currently *observed* labeled bars, persist its complete transformation pipeline, and write next-bar predictions **before outcomes arrive**. The historical final holdout is already closed; these are new prospective predictions, not another holdout evaluation.

**Important:** Run after your daily OHLCV ingestion. Ensure the most recent bar for each ticker is **complete** before predicting. The notebook cannot automatically verify an in-progress intraday candle. Use the same interval (`1d`) throughout. Running twice on unchanged data is idempotent for ledger keys. Model versions are immutable and training is not repeated unless you explicitly opt into a new version.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from stock_ml_dynamic.db import engine
from stock_ml_dynamic.master_table import fetch_universe_prices
from stock_ml_dynamic.prospective import (prepare_universe, train_frozen_model, save_model,
    load_model, predict_latest, append_predictions, reconcile_predictions, prospective_scores)

INTERVAL = '1d'
ARTIFACTS = ROOT / 'artifacts' / 'prospective' / INTERVAL
MODELS = ARTIFACTS / 'models'
LEDGER = ARTIFACTS / 'prediction_ledger.parquet'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print('Repository:', ROOT)
print('Ledger:', LEDGER)

## 1 · Refresh data from PostgreSQL

This is deliberately **not** the historical `master_labeled.parquet`: predictions require the latest observed OHLCV bars. Keep the data fetch read-only. If your ingestion process includes an unfinished bar, wait for it to complete.

In [ ]:
with engine().connect() as conn:
    raw_prices = fetch_universe_prices(conn, interval=INTERVAL, bars_per_symbol=2000)
print('Raw bars:', len(raw_prices), 'tickers:', raw_prices.symbol.nunique())
print('Newest timestamp:', raw_prices.bar_timestamp.max())
frame, extra, eligibility = prepare_universe(raw_prices)
print('Eligible tickers:', frame.symbol.nunique(), 'of', eligibility.symbol.nunique())
display(eligibility.loc[~eligibility.eligible, ['symbol','bars','latest_adj_z','eligible']].head(15))

## 2 · Persist frozen model (explicit training switch)

On first execution, set `TRAIN_NEW_MODEL=True`. After saving, change it to `False` for subsequent daily prediction runs. Model versions derive from the latest **observed training outcome timestamp**, so attempting to overwrite an existing version fails safely. If a new daily bar arrives, explicitly retrain to create a new model version; for a fixed-model prospective trial, keep using the original saved version.

In [ ]:
TRAIN_NEW_MODEL = True  # Set False for daily inference after the first run.
MODEL_PATH = None  # When TRAIN_NEW_MODEL=False, set to the existing model.joblib path.

if TRAIN_NEW_MODEL:
    bundle = train_frozen_model(frame, extra, max_train_rows=120000, selected_k=4)
    model_path = save_model(bundle, MODELS)
    print('Saved model:', model_path)
else:
    if MODEL_PATH is None:
        candidates = sorted(MODELS.glob('*/model.joblib'))
        if not candidates:
            raise FileNotFoundError('No saved model; train once first.')
        model_path = candidates[0]  # Oldest frozen model: keep a fixed prospective trial.
    else:
        model_path = Path(MODEL_PATH)
    bundle = load_model(model_path)
    print('Loaded frozen model:', model_path)
print('Version:', bundle['model_version'], 'fit rows:', bundle['train_rows'])

## 3 · Record timestamped next-bar predictions

The ledger stores each ticker, model version, as-of bar, class probabilities, prediction, training-fitted tertile boundaries, and the time the prediction was recorded. Existing keys are never rewritten. **Do not backfill predictions for bars that already have a known next outcome and call them prospective.**

In [ ]:
predictions = predict_latest(frame, bundle)
print('Inference-ready:', len(predictions), 'tickers:', predictions.symbol.nunique())
display(predictions[['symbol','bar_timestamp','predicted_direction','prob_0','prob_1','prob_2']].head(12))

# Confirm the current bar is final/complete and that the next bar has not arrived.
# Explicit opt-in prevents accidentally recording stale or retrospective predictions.
RECORD_PREDICTIONS = False  # Change to True only after verifying current data freshness.
if RECORD_PREDICTIONS:
    added, total = append_predictions(predictions, LEDGER)
    print(f'Appended {added} new predictions; ledger has {total} total rows.')
else:
    print('Preview only; set RECORD_PREDICTIONS=True to commit future predictions.')

## 4 · Resolve previously recorded predictions after ingestion

Only after the *next* bar appears, compare the original prediction against its first subsequently observed adjusted close. The saved label boundaries are reused; probabilities remain unchanged. **Do not use this cell to retroactively score newly generated predictions**; generate predictions before their future bars exist.

In [ ]:
if LEDGER.exists():
    ledger, resolved_now = reconcile_predictions(LEDGER, raw_prices)
    print('Newly resolved:', resolved_now, 'total resolved:', ledger.actual_direction.notna().sum())
    display(prospective_scores(ledger, min_rows=30))
else:
    print('No ledger yet; record prospective predictions after validating bar completeness.')

## 5 · Operating rules

- **Do not interpret the previous 37.83% holdout accuracy as a live result.** Prospective performance starts only after predictions are recorded before their outcomes.
- Keep the first frozen model version for a clean prospective trial. A later retrain is a **new** model version, evaluated separately.
- This file-based ledger is intended for a **single writer**; do not run simultaneous notebook sessions writing it.
- For daily bars, confirm data ingestion and candle finality before prediction; holidays and missing bars can delay resolution.
- The saved `joblib` artifact uses pickle serialization: load **only locally trusted** files.
- These are paper predictions, not orders or investment advice.